## 13.10 预训练BERT


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()
from src.utils import (load_data_wiki, get_tokens_and_segments,
                       Timer, Accumulator)


### 练习 13.10.1

**题目：** 在实验中，我们可以看到遮蔽语言模型损失明显高于下一句预测损失。为什么？

**解答：** 两类任务的“难度与粒度”不同：

- **掩蔽语言模型（MLM）**：需要从双向上下文预测被掩蔽的**多个词元**，每个词元都要在 `len(vocab)`（约 2 万）个候选词中做分类，输出维度大、单点难度高，损失是**逐词元**的交叉熵之和，自然数值更大；
- **下一句预测（NSP）**：只需把 `[CLS]` 位置的表示送入二分类层判断“是否下一句”，输出只有 2 类，任务简单，损失小得多。

从主 notebook 的训练曲线看，MLM 损失大约在 3~4 的量级，而 NSP 损失在 0.5~0.8 左右，差距正来自上述任务粒度与分类难度差异。实践中也可对两个损失加权求和来平衡量纲。


### 练习 13.10.2

**题目：** 将 BERT 输入序列的最大长度设置为 512（与原始 BERT 模型相同）。使用原始 BERT 模型的配置，如 $\text{BERT}_{\text{LARGE}}$。运行此部分时是否遇到错误？为什么？

**解答：** 会**遇到内存不足（OOM）错误**，原因如下：

1. **序列长度增大**：`max_len=512` 时，注意力矩阵形状为 $(\text{batch}, \text{heads}, 512, 512)$，相比 `max_len=64` 时的 $(\cdot, \cdot, 64, 64)$，注意力显存占用放大 $(512/64)^2 = 64$ 倍；
2. **模型规模增大**：$\text{BERT}_{\text{LARGE}}$ 有 24 个 Transformer 块、16 头、1024 隐藏维、约 3.4 亿参数，而主 notebook 的示例模型仅 2 层、隐藏维 64；
3. **MLM 输出头**：逐词元在 `len(vocab)` 上分类，词表 2 万 × 512 位置的 logits 也占用可观显存；
4. **训练需要反向传播**：保存中间激活用于求梯度，进一步放大内存需求。

在主 notebook 环境（单卡 NPU）下直接运行必然 OOM；需要减小 batch size、梯度累积、混合精度（AMP）或使用更大的显存才能跑通。这也解释了为什么原始 BERT 预训练要用多卡 TPU/GPU 集群。

下面用一个小规模验证：加载 WikiText-2 并把 `max_len` 调大观察单批内存变化趋势（实际 OOM 演示会拖垮环境，这里仅以 `max_len` 对注意力张量大小的放大倍数说明）：


In [2]:
# 注意力矩阵大小随 max_len 的放大倍数（不实际分配，仅计算）
import math
for max_len in (64, 128, 256, 512):
    attn = max_len * max_len          # 单头注意力矩阵元素数
    attn_64 = 64 * 64
    print(f'max_len={max_len:4d}: 注意力矩阵为 {max_len}x{max_len}, '
          f'是 max_len=64 的 {attn / attn_64:.0f} 倍')
print()
print('BERT_LARGE 配置：24 层、1024 隐藏维、16 头、约 3.4 亿参数，'
      '相对主 notebook 示例模型（2 层、64 隐藏维）参数多约 500 倍，'
      '加上 512 长度注意力的 64 倍放大，单卡 NPU 必然内存不足。')


max_len=  64: 注意力矩阵为 64x64, 是 max_len=64 的 1 倍
max_len= 128: 注意力矩阵为 128x128, 是 max_len=64 的 4 倍
max_len= 256: 注意力矩阵为 256x256, 是 max_len=64 的 16 倍
max_len= 512: 注意力矩阵为 512x512, 是 max_len=64 的 64 倍

BERT_LARGE 配置：24 层、1024 隐藏维、16 头、约 3.4 亿参数，相对主 notebook 示例模型（2 层、64 隐藏维）参数多约 500 倍，加上 512 长度注意力的 64 倍放大，单卡 NPU 必然内存不足。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
